# Initial Setup

In [1]:
import polars as pl 
import pandas as pd
from utils.general_purpose import get_feature_definitions, count_missing_data
from utils.validation import calculate_gini_stability_metric

In [2]:
# VARIABLES
TRAIN_DATA_PATH = "../data/csv_files/train/"
TEST_DATA_PATH = "../data/csv_files/test/"
FEATURE_DEF_PATH = "../data/feature_definitions.csv"

# With polars
POLARS_TRAIN_DATA_PATH = "../data/train/"
POLARS_TEST_DATA_PATH = "../data/test/"

In [3]:
# Set polars config
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(-1)
# Set pandas config (the utils is made by pandas so we need this too)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Business Understanding

## Describtion

Consumer finance providers must accurately determine which clients can repay a loan and which cannot and data is key. If data science could help better predict one’s repayment capabilities, loans might become more accessible to those who may benefit from them the most.

**The Problem**: The scorecard's stability in the future is critical, as a sudden drop in performance means that loans will be issued to worse clients on average. The core of the issue is that **loan providers aren't able to spot potential problems any sooner than the first due dates of those loans are observable**. Given the time it takes to redevelop, validate, and implement the scorecard, **stability is highly desirable**. 

**Keyword**: stability, accuracy

## Evaluation

The subscription is using the gini stability metric. A gini score is calculated for predictions corresponding to each WEEK_NUM.
$$
    \text{Gini} = 2 \times \text{AUC} - 1
$$
A linear regression, $f(x) = a \cdot x + b$ is fit through the weekly gini score, and a falling_rate is calculated as $min(0,a)$. This is used to penalize models that drop off in predictive ability.

Finally, the variability of the predictions are calculated by taking the standard deviation of the residuals from the above linear regression, applying a penalty to model variablity.

Final metric is calculated as: 
$$
    \text{stability metric} = \mu(gini) + 88.0 \cdot min(0,a) - 0.5 \cdot \sigma(residuals)
$$




# Data Understanding

## Data Depth 0

### Train Base

In [4]:
# Get the columns 
train_base_feature = get_feature_definitions(data_file_path=TRAIN_DATA_PATH+"train_base.csv",
                                             feature_definitions_csv=FEATURE_DEF_PATH)
train_base_feature

,Variable,Description,Data Type


In [5]:
# Read train base 
train_base = pl.read_parquet(POLARS_TRAIN_DATA_PATH+"train_base.parquet")

In [6]:
train_base.head()

case_id,date_decision,MONTH,WEEK_NUM,target
i64,str,i64,i64,i64
0,"""2019-01-03""",201901,0,0
1,"""2019-01-03""",201901,0,0
2,"""2019-01-04""",201901,0,0
3,"""2019-01-03""",201901,0,0
4,"""2019-01-04""",201901,0,1


In [7]:
train_base.describe()

statistic,case_id,date_decision,MONTH,WEEK_NUM,target
str,f64,str,f64,f64,f64
"""count""",1.526659e6,"""1526659""",1.526659e6,1.526659e6,1.526659e6
"""null_count""",0.0,"""0""",0.0,0.0,0.0
"""mean""",1.2861e6,null,201936.287982,40.769036,0.031437
"""std""",718946.592285,null,44.735975,23.797981,0.174496
"""min""",0.0,"""2019-01-01""",201901.0,0.0,0.0
"""25%""",766198.0,null,201906.0,23.0,0.0
"""50%""",1.357358e6,null,201910.0,40.0,0.0
"""75%""",1.739023e6,null,202001.0,55.0,0.0
"""max""",2.703454e6,"""2020-10-05""",202010.0,91.0,1.0


Insights:
- There is no null data
- Unique case ID (which is ok)
- Around 2 years worth of data, 
- Lots of week num, 91 week as max,
- Target is binary variable - if 0 then no default, if 1 then default 


**Conclusion**: This table is like a fact table.

### Train static dataset

Remember that the train static is the internal dataset

In [8]:
# Get the feature definitions 
train_static_feature = get_feature_definitions(data_file_path=TRAIN_DATA_PATH+"train_static_0_0.csv",
                                             feature_definitions_csv=FEATURE_DEF_PATH)
train_static_feature

,Variable,Description,Data Type
1,actualdpdtolerance_344P,DPD of client with tolerance.,float64
14,amtinstpaidbefduel24m_4187115A,Number of instalments paid before due date in the last 24 months.,float64
17,annuity_780A,Monthly annuity amount.,float64
19,annuitynextmonth_57A,Next month's amount of annuity.,float64
20,applicationcnt_361L,Number of applications associated with the same email address as the client.,float64
21,applications30d_658L,Number of applications made by the client in the last 30 days.,float64
22,applicationscnt_1086L,Number of applications associated with the same phone number.,float64
23,applicationscnt_464L,Number of applications made in the last 30 days by other clients with the same employer as the applicant.,float64
24,applicationscnt_629L,Number of applications with the same employer in the last 7 days.,float64
25,applicationscnt_867L,Number of applications associated with the same mobile phone.,float64


This is the DEPTH 0 type, which mean that this is the feature that can directly be used in machine learning model

In [9]:
train_static_0 = pl.read_parquet(POLARS_TRAIN_DATA_PATH+"train_static_0_0.parquet")
print(train_static_0.shape)
train_static_1 = pl.read_parquet(POLARS_TRAIN_DATA_PATH+"train_static_0_1.parquet")
print(train_static_1.shape) 

(1003757, 168)
(522902, 168)


Now let concat these 2 tables

In [10]:
train_static = pl.concat([train_static_0, train_static_1], how="vertical")
train_static.shape

(1526659, 168)

The value here is the same as the train_base, which mean that we can join them. 

In [11]:
train_static.describe()

statistic,case_id,actualdpdtolerance_344P,amtinstpaidbefduel24m_4187115A,annuity_780A,annuitynextmonth_57A,applicationcnt_361L,applications30d_658L,applicationscnt_1086L,applicationscnt_464L,applicationscnt_629L,applicationscnt_867L,avgdbddpdlast24m_3658932P,avgdbddpdlast3m_4187120P,avgdbdtollast24m_4525197P,avgdpdtolclosure24_3658938P,avginstallast24m_3658937A,avglnamtstart24m_4525187A,avgmaxdpdlast9m_3716943P,avgoutstandbalancel6m_4187114A,avgpmtlast12m_4525200A,bankacctype_710L,cardtype_51L,clientscnt12m_3712952L,clientscnt3m_3712950L,clientscnt6m_3712949L,clientscnt_100L,clientscnt_1022L,clientscnt_1071L,clientscnt_1130L,clientscnt_136L,clientscnt_157L,clientscnt_257L,clientscnt_304L,clientscnt_360L,clientscnt_493L,clientscnt_533L,clientscnt_887L,clientscnt_946L,cntincpaycont9m_3716944L,cntpmts24_3658933L,commnoinclast6m_3546845L,credamount_770A,credtype_322L,currdebt_22A,currdebtcredtyperange_828A,datefirstoffer_1144D,datelastinstal40dpd_247D,datelastunpaid_3546854D,daysoverduetolerancedd_3976961L,deferredmnthsnum_166L,disbursedcredamount_1113A,disbursementtype_67L,downpmt_116A,dtlastpmtallstes_4499206D,eir_270L,equalitydataagreement_891L,equalityempfrom_62L,firstclxcampaign_1125D,firstdatedue_489D,homephncnt_628L,inittransactionamount_650A,inittransactioncode_186L,interestrate_311L,interestrategrace_34L,isbidproduct_1095L,isbidproductrequest_292L,isdebitcard_729L,lastactivateddate_801D,lastapplicationdate_877D,lastapprcommoditycat_1041M,lastapprcommoditytypec_5251766M,lastapprcredamount_781A,lastapprdate_640D,lastcancelreason_561M,lastdelinqdate_224D,lastdependentsnum_448L,lastotherinc_902A,lastotherlnsexpense_631A,lastrejectcommoditycat_161M,lastrejectcommodtypec_5251769M,lastrejectcredamount_222A,lastrejectdate_50D,lastrejectreason_759M,lastrejectreasonclient_4145040M,lastrepayingdate_696D,lastst_736L,maininc_215A,mastercontrelectronic_519L,mastercontrexist_109L,maxannuity_159A,maxannuity_4075009A,maxdbddpdlast1m_3658939P,maxdbddpdtollast12m_3658940P,maxdbddpdtollast6m_4187119P,maxdebt4_972A,maxdpdfrom6mto36m_3546853P,maxdpdinstldate_3546855D,maxdpdinstlnum_3546846P,maxdpdlast12m_727P,maxdpdlast24m_143P,maxdpdlast3m_392P,maxdpdlast6m_474P,maxdpdlast9m_1059P,maxdpdtolerance_374P,maxinstallast24m_3658928A,maxlnamtstart6m_4525199A,maxoutstandbalancel12m_4187113A,maxpmtlast3m_4525190A,mindbddpdlast24m_3658935P,mindbdtollast24m_4525191P,mobilephncnt_593L,monthsannuity_845L,numactivecreds_622L,numactivecredschannel_414L,numactiverelcontr_750L,numcontrs3months_479L,numincomingpmts_3546848L,numinstlallpaidearly3d_817L,numinstls_657L,numinstlsallpaid_934L,numinstlswithdpd10_728L,numinstlswithdpd5_4187116L,numinstlswithoutdpd_562L,numinstmatpaidtearly2d_4499204L,numinstpaid_4499208L,numinstpaidearly3d_3546850L,numinstpaidearly3dest_4493216L,numinstpaidearly5d_1087L,numinstpaidearly5dest_4493211L,numinstpaidearly5dobd_4499205L,numinstpaidearly_338L,numinstpaidearlyest_4493214L,numinstpaidlastcontr_4325080L,numinstpaidlate1d_3546852L,numinstregularpaid_973L,numinstregularpaidest_4493210L,numinsttopaygr_769L,numinsttopaygrest_4493213L,numinstunpaidmax_3546851L,numinstunpaidmaxest_4493212L,numnotactivated_1143L,numpmtchanneldd_318L,numrejects9m_859L,opencred_647L,paytype1st_925L,paytype_783L,payvacationpostpone_4187118D,pctinstlsallpaidearl3d_427L,pctinstlsallpaidlat10d_839L,pctinstlsallpaidlate1d_3546856L,pctinstlsallpaidlate4d_3546849L,pctinstlsallpaidlate6d_3546844L,pmtnum_254L,posfpd10lastmonth_333P,posfpd30lastmonth_3976960P,posfstqpd30lastmonth_3976962P,previouscontdistrict_112M,price_1097A,sellerplacecnt_915L,sellerplacescnt_216L,sumoutstandtotal_3546847A,sumoutstandtotalest_4493215A,totaldebt_9A,totalsettled_863A,totinstallast1m_4525188A,twobodfilling_608L,typesuite_864L,validfrom_1069D
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,str,str,str,f64,f64,f64,str,f64,str,f64,f64,f64,str,str,f6

In [12]:
train_static_missing_stats = count_missing_data(df=train_static)
train_static_missing_stats

,Column Name,Missing Count,Missing Percentage
0,clientscnt_136L,1526242,99.972685
1,lastrepayingdate_696D,1524282,99.844301
2,lastotherinc_902A,1523603,99.799824
3,lastotherlnsexpense_631A,1523601,99.799693
4,payvacationpostpone_4187118D,1517330,99.388927
5,isbidproductrequest_292L,1514201,99.183970
6,interestrategrace_34L,1505776,98.632111
7,lastdependentsnum_448L,1497142,98.066562
8,equalityempfrom_62L,1488847,97.523219
9,maxannuity_4075009A,1450726,95.026198


Every feature that is missing more than 90% can be considered as redundant. Therefore we can remove them

In [13]:
from typing import Union
def missing_data_cutoff_with_threshold(df: Union[pd.DataFrame, pl.DataFrame], threshold: float) -> Union[pd.DataFrame, pl.DataFrame]:
    """
    Eliminates columns from a DataFrame (Pandas or Polars) that have a missingness
    percentage exceeding a specified threshold.

    Parameters:
    -----------
    df : Union[pd.DataFrame, pl.DataFrame]
        The input DataFrame, either a Pandas DataFrame or a Polars DataFrame.
    threshold : float
        The percentage threshold (e.g., 90.0 for 90%) above which columns
        will be dropped.

    Returns:
    --------
    Union[pd.DataFrame, pl.DataFrame]
        A new DataFrame with columns exceeding the threshold removed.
        The returned DataFrame will be of the same type as the input.
    """
    if not (0 <= threshold <= 100):
        raise ValueError("Threshold must be a percentage between 0 and 100.")

    # Get the missing data summary using our previous function
    missing_summary_df = count_missing_data(df)

    # Identify columns to drop based on the threshold
    columns_to_drop = missing_summary_df[
        missing_summary_df['Missing Percentage'] > threshold
    ]['Column Name'].tolist()

    if not columns_to_drop:
        print(f"No columns found with missing percentage above {threshold}%.")
        return df.copy() # Return a copy to avoid modifying the original DataFrame in place

    print(f"Dropping {len(columns_to_drop)} columns with missing percentage above {threshold}%:")
    for col in columns_to_drop:
        print(f"- {col}")

    # Perform the drop operation based on DataFrame type
    if isinstance(df, pd.DataFrame):
        new_df = df.drop(columns=columns_to_drop)
    elif isinstance(df, pl.DataFrame):
        # Polars has a direct .drop() method
        new_df = df.drop(columns_to_drop)
    else:
        # This case should ideally not be reached if count_missing_data also validated input
        raise TypeError("Input must be either a pandas.DataFrame or a polars.DataFrame.")

    return new_df

In [14]:
train_static_remove_missing = missing_data_cutoff_with_threshold(df=train_static, threshold=90.0)
train_static_remove_missing.shape

Dropping 13 columns with missing percentage above 90.0%:
- clientscnt_136L
- lastrepayingdate_696D
- lastotherinc_902A
- lastotherlnsexpense_631A
- payvacationpostpone_4187118D
- isbidproductrequest_292L
- interestrategrace_34L
- lastdependentsnum_448L
- equalityempfrom_62L
- maxannuity_4075009A
- equalitydataagreement_891L
- datelastinstal40dpd_247D
- validfrom_1069D


(1526659, 155)

In [16]:
train_static_remove_missing_stats = count_missing_data(df=train_static_remove_missing)
train_static_remove_missing_stats

,Column Name,Missing Count,Missing Percentage
0,avglnamtstart24m_4525187A,1364150,89.355252
1,cardtype_51L,1334968,87.443758
2,isdebitcard_729L,1334357,87.403736
3,inittransactionamount_650A,1334357,87.403736
4,totinstallast1m_4525188A,1174211,76.913771
5,maxpmtlast3m_4525190A,1129330,73.973952
6,typesuite_864L,1121505,73.461395
7,bankacctype_710L,1109629,72.683487
8,maxlnamtstart6m_4525199A,1032856,67.654663
9,avgpmtlast12m_4525200A,1026987,67.270229


# EDA